In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model

In [25]:
# 1. Convolution block
def conv_block(x, filters):
    x = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x


# 2. Max pooling with indices
class MaxPoolWithArgmax(layers.Layer):
    def call(self, x):
        pooled, indices = tf.nn.max_pool_with_argmax(
            x,
            ksize=2,
            strides=2,
            padding="VALID",
            output_dtype=tf.int64,
            include_batch_in_index=True
        )
        return pooled, indices

    def compute_output_shape(self, input_shape):
        batch, height, width, channels = input_shape

        pooled_shape = (
            batch,
            height // 2,
            width // 2,
            channels
        )
        return pooled_shape, pooled_shape

In [26]:

# 3. Unpooling using saved indices
class MaxUnpool(layers.Layer):
    def call(self, inputs):
        pooled, indices, reference = inputs

        output_shape = tf.shape(reference)

        flat_indices = tf.reshape(indices, [-1, 1])
        flat_values = tf.reshape(pooled, [-1])

        output = tf.scatter_nd(
            flat_indices,
            flat_values,
            tf.reshape(tf.size(reference), [1])
        )

        return tf.reshape(output, output_shape)

    def compute_output_shape(self, input_shape):
        return input_shape[2]

In [27]:
# 4. SegNet architecture
def segnet(input_shape=(128, 128, 3), num_classes=11):
    inputs = layers.Input(shape=input_shape)

    pool = MaxPoolWithArgmax()
    unpool = MaxUnpool()

    # ---------- ENCODER ----------

    # Encoder block 1
    e1 = conv_block(inputs, 64)
    e1 = conv_block(e1, 64)
    p1, i1 = pool(e1)

    # Encoder block 2
    e2 = conv_block(p1, 128)
    e2 = conv_block(e2, 128)
    p2, i2 = pool(e2)

    # Encoder block 3
    e3 = conv_block(p2, 256)
    e3 = conv_block(e3, 256)
    e3 = conv_block(e3, 256)
    p3, i3 = pool(e3)

    # Encoder block 4
    e4 = conv_block(p3, 512)
    e4 = conv_block(e4, 512)
    e4 = conv_block(e4, 512)
    p4, i4 = pool(e4)

    # Encoder block 5
    e5 = conv_block(p4, 512)
    e5 = conv_block(e5, 512)
    e5 = conv_block(e5, 512)
    p5, i5 = pool(e5)

    # ---------- DECODER ----------

    # Decoder block 5
    d5 = unpool([p5, i5, e5])
    d5 = conv_block(d5, 512)
    d5 = conv_block(d5, 512)
    d5 = conv_block(d5, 512)

    # Decoder block 4
    d4 = unpool([d5, i4, e4])
    d4 = conv_block(d4, 512)
    d4 = conv_block(d4, 512)
    d4 = conv_block(d4, 256)

    # Decoder block 3
    d3 = unpool([d4, i3, e3])
    d3 = conv_block(d3, 256)
    d3 = conv_block(d3, 256)
    d3 = conv_block(d3, 128)

    # Decoder block 2
    d2 = unpool([d3, i2, e2])
    d2 = conv_block(d2, 128)
    d2 = conv_block(d2, 64)

    # Decoder block 1
    d1 = unpool([d2, i1, e1])
    d1 = conv_block(d1, 64)

    # ---------- PIXEL CLASSIFIER ----------

    outputs = layers.Conv2D(
        num_classes,
        kernel_size=1,
        padding="same",
        activation=None
    )(d1)

    return Model(inputs, outputs, name="SegNet")

In [28]:
model = segnet()
model.summary()


Model: "SegNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_48 (Conv2D)  │ (None, 128, 128,  │      1,728 │ input_layer_7[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_48[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_47 (ReLU)     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_49 (Conv2D)  │ (None, 128, 128,  │     36,864 │ re_lu_47[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_49[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_48 (ReLU)     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pool_with_argm… │ [(None, 4, 4,     │          0 │ re_lu_48[0][0],   │
│ (MaxPoolWithArgmax) │ 512), (None, 4,   │            │ re_lu_50[0][0],   │
│                     │ 4, 512)]          │            │ re_lu_53[0][0],   │
│                     │                   │            │ re_lu_56[0][0],   │
│                     │                   │            │ re_lu_59[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_50 (Conv2D)  │ (None, 64, 64,    │     73,728 │ max_pool_with_ar… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_50[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_49 (ReLU)     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_51 (Conv2D)  │ (None, 64, 64,    │    147,456 │ re_lu_49[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_51[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_50 (ReLU)     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_52 (Conv2D)  │ (None, 32, 32,    │    294,912 │ max_pool_with_ar… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │      1,024 │ conv2d_52[0][0] 

 Total params: 29,451,659 (112.35 MB)

 Trainable params: 29,435,787 (112.29 MB)

 Non-trainable params: 15,872 (62.00 KB)

In [29]:
print("Input shape", model.input_shape)
print("Output Shape", model.output_shape)

Input shape (None, 128, 128, 3)
Output Shape (None, 128, 128, 11)


In [30]:
print(
    "Output spatial dimensions restored:",
    model.output_shape[1:3] == model.input_shape[1:3]
)

Output spatial dimensions restored: True
